In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""앙상블 Walk-Forward v2 (메타모델 개선)
v1 대비 변경:
  A. 메타모델 피처 확장: tft_prob, lgbm_prob + 불일치도, 합의강도, vol_5d, risk_flag
  B. 슬라이딩 윈도우 학습: 전체 이력 → 최근 4윈도우(1년)만 사용
  C. 메타모델 비교: LogisticRegression vs LightGBM 메타모델

체크포인트 의존:
  models/tft_v2_wf/checkpoints/window_XX.ckpt
  models/lgbm_wf/checkpoints/window_XX.joblib
  models/garch_wf/checkpoints/window_XX.parquet
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_wf_v2"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# ===== v2 메타모델 설정 =====
SLIDING_WINDOW = 4   # 최근 N윈도우만 학습 (0=전체)
META_FEATURES = ["tft_prob", "lgbm_prob", "disagreement", "consensus_strength", "vol_5d", "risk_flag"]

# 윈도우 생성
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"Walk-Forward 윈도우: {len(windows)}개")
print(f"v2 개선: 피처 {len(META_FEATURES)}개, 슬라이딩 윈도우={SLIDING_WINDOW}")
print(f"\n체크포인트 확인:")
for dir_name, ckpt_dir in [("TFT", TFT_CKPT_DIR), ("LightGBM", LGBM_CKPT_DIR), ("GARCH", GARCH_CKPT_DIR)]:
    if ckpt_dir.exists():
        n = len(list(ckpt_dir.iterdir()))
        print(f"  {dir_name}: {n}개 체크포인트")
    else:
        print(f"  {dir_name}: 디렉토리 없음!")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): x,y=b; return self.loss_fn(self(x),y)
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 모델 정의 완료 (로드 전용)")

In [ ]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    """TFT용 시퀀스 샘플 생성 (date, ticker 메타 포함)."""
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    """TFT 모델로 예측."""
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    """LightGBM 모델로 예측."""
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

In [ ]:
# ===== Step 1: 모든 윈도우의 예측 수집 (v1과 동일) =====
all_predictions = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    if not tft_ckpt.exists() or not lgbm_ckpt.exists():
        print(f"  [{i:2d}] SKIP: 체크포인트 없음"); continue
    
    print(f"  [{i:2d}] {test_start}~{test_end} 예측 생성...")
    
    tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
    tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    if len(tft_samples) < 100:
        del tft_model; torch.cuda.empty_cache(); continue
    tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
    del tft_model; torch.cuda.empty_cache()
    
    lgbm_model = joblib.load(str(lgbm_ckpt))
    lgbm_probs, _, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
    del lgbm_model
    
    garch_df = pd.read_parquet(str(garch_ckpt)) if garch_ckpt.exists() else None
    
    tft_df = pd.DataFrame(tft_metas); tft_df["tft_prob"] = tft_probs; tft_df["label"] = tft_labels
    lgbm_df = pd.DataFrame(lgbm_metas); lgbm_df["lgbm_prob"] = lgbm_probs
    merged = tft_df.merge(lgbm_df, on=["date","ticker"], how="inner")
    merged["window"] = i
    
    if garch_df is not None and "ticker" in garch_df.columns:
        merged = merged.merge(garch_df[["ticker","vol_5d","risk_flag"]], on="ticker", how="left")
        merged["vol_5d"] = merged["vol_5d"].fillna(merged["vol_5d"].median() if merged["vol_5d"].notna().any() else 40.0)
        merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
    else:
        merged["vol_5d"] = 40.0; merged["risk_flag"] = 0
    
    # v2: 파생 피처 생성
    merged["disagreement"] = (merged["tft_prob"] - merged["lgbm_prob"]).abs()
    merged["consensus_strength"] = merged["tft_prob"] * merged["lgbm_prob"]
    
    all_predictions.append(merged)
    print(f"       {len(merged):,} 예측")

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f"\n전체 예측: {len(pred_df):,} rows, 윈도우 {pred_df['window'].nunique()}개")
print(f"메타 피처: {META_FEATURES}")

In [ ]:
# ===== Step 2: 메타모델 비교 (v1 vs v2 개선) =====
# 6가지 앙상블 방법을 윈도우별로 평가

BT_START = 1  # Window 0은 학습 전용

methods = {
    "tft_only": [],
    "lgbm_only": [],
    "v1_lr_all": [],        # v1: LR + 2피처 + 전체이력
    "v2_lr_slide": [],      # v2-A+B: LR + 6피처 + 슬라이딩
    "v2_lgbm_slide": [],    # v2-A+B+C: LightGBM + 6피처 + 슬라이딩
    "v2_lgbm_all": [],      # v2-A+C: LightGBM + 6피처 + 전체이력
}

wf_details = []

for i in range(BT_START, pred_df["window"].max() + 1):
    test_mask = pred_df["window"] == i
    if test_mask.sum() == 0: continue
    
    y_true = pred_df[test_mask]["label"].values
    test_X_basic = pred_df[test_mask][["tft_prob", "lgbm_prob"]].values
    test_X_full = pred_df[test_mask][META_FEATURES].values
    
    row = {"window": i, "test_start": windows[i][1], "test_end": windows[i][2],
           "n_samples": int(test_mask.sum())}
    
    # 개별 모델
    for name, col in [("tft_only", "tft_prob"), ("lgbm_only", "lgbm_prob")]:
        probs = pred_df[test_mask][col].values
        da = accuracy_score(y_true, (probs >= 0.5).astype(int)) * 100
        methods[name].append(da)
        row[name] = round(da, 2)
    
    # --- v1: LR + 2피처 + 전체이력 ---
    train_v1 = pred_df["window"] < i
    if train_v1.sum() >= 100:
        lr_v1 = LogisticRegression(random_state=42, class_weight="balanced")
        lr_v1.fit(pred_df[train_v1][["tft_prob","lgbm_prob"]].values, pred_df[train_v1]["label"].values)
        probs = lr_v1.predict_proba(test_X_basic)[:, 1]
        da = accuracy_score(y_true, (probs >= 0.5).astype(int)) * 100
        methods["v1_lr_all"].append(da)
        row["v1_lr_all"] = round(da, 2)
        row["v1_tft_w"] = round(lr_v1.coef_[0][0], 2)
        row["v1_lgbm_w"] = round(lr_v1.coef_[0][1], 2)
    
    # --- v2-A+B: LR + 6피처 + 슬라이딩 ---
    train_slide = (pred_df["window"] >= max(0, i - SLIDING_WINDOW)) & (pred_df["window"] < i)
    if train_slide.sum() >= 100:
        lr_v2 = LogisticRegression(random_state=42, class_weight="balanced", max_iter=500)
        lr_v2.fit(pred_df[train_slide][META_FEATURES].values, pred_df[train_slide]["label"].values)
        probs = lr_v2.predict_proba(test_X_full)[:, 1]
        da = accuracy_score(y_true, (probs >= 0.5).astype(int)) * 100
        methods["v2_lr_slide"].append(da)
        row["v2_lr_slide"] = round(da, 2)
    
    # --- v2-A+B+C: LightGBM 메타 + 6피처 + 슬라이딩 ---
    if train_slide.sum() >= 100:
        meta_lgbm = lgb.LGBMClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced", random_state=42, verbose=-1)
        meta_lgbm.fit(pred_df[train_slide][META_FEATURES].values, pred_df[train_slide]["label"].values)
        probs = meta_lgbm.predict_proba(test_X_full)[:, 1]
        da = accuracy_score(y_true, (probs >= 0.5).astype(int)) * 100
        methods["v2_lgbm_slide"].append(da)
        row["v2_lgbm_slide"] = round(da, 2)
    
    # --- v2-A+C: LightGBM 메타 + 6피처 + 전체이력 ---
    train_all = pred_df["window"] < i
    if train_all.sum() >= 100:
        meta_lgbm2 = lgb.LGBMClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced", random_state=42, verbose=-1)
        meta_lgbm2.fit(pred_df[train_all][META_FEATURES].values, pred_df[train_all]["label"].values)
        probs = meta_lgbm2.predict_proba(test_X_full)[:, 1]
        da = accuracy_score(y_true, (probs >= 0.5).astype(int)) * 100
        methods["v2_lgbm_all"].append(da)
        row["v2_lgbm_all"] = round(da, 2)
    
    wf_details.append(row)
    print(f"  [{i:2d}] TFT={row.get('tft_only','?'):.1f} LGBM={row.get('lgbm_only','?'):.1f} "
          f"v1_LR={row.get('v1_lr_all','?')} "
          f"v2_LR_s={row.get('v2_lr_slide','?')} "
          f"v2_LGBM_s={row.get('v2_lgbm_slide','?')} "
          f"v2_LGBM_a={row.get('v2_lgbm_all','?')}")

print(f"\n평가 완료: {len(wf_details)}개 윈도우")

In [ ]:
# ===== Step 3: 결과 비교 =====
print("="*85)
print("  앙상블 메타모델 비교 (v1 vs v2)")
print("="*85)

labels = {
    "tft_only": "TFT v2 단독",
    "lgbm_only": "LightGBM 단독",
    "v1_lr_all": "v1: LR(2피처,전체)",
    "v2_lr_slide": "v2: LR(6피처,슬라이딩)",
    "v2_lgbm_slide": "v2: LGBM(6피처,슬라이딩)",
    "v2_lgbm_all": "v2: LGBM(6피처,전체)",
}

print(f"\n  {'방법':28s} {'평균DA':>8s} {'Std':>7s} {'Min':>7s} {'Max':>7s} {'윈도우':>6s}")
print(f"  {'-'*66}")
for name in methods:
    das = methods[name]
    if das:
        print(f"  {labels[name]:28s} {np.mean(das):>7.2f}% {np.std(das):>6.2f} {min(das):>6.2f}% {max(das):>6.2f}% {len(das):>5d}")

# 최적 방법
best = max(methods, key=lambda k: np.mean(methods[k]) if methods[k] else 0)
print(f"\n  최적: {labels[best]} (평균 DA={np.mean(methods[best]):.2f}%)")

# v1 vs v2 개선폭
if methods["v1_lr_all"] and methods["v2_lgbm_slide"]:
    v1_mean = np.mean(methods["v1_lr_all"])
    v2_mean = np.mean(methods["v2_lgbm_slide"])
    print(f"  v1→v2 개선: {v1_mean:.2f}% → {v2_mean:.2f}% ({v2_mean-v1_mean:+.2f}%p)")

# 윈도우별 상세
print(f"\n  {'#':>3} {'기간':>24} {'TFT':>6} {'LGBM':>6} {'v1_LR':>6} {'v2_LRs':>6} {'v2_Ls':>6} {'v2_La':>6} {'최고':>8}")
print(f"  {'-'*82}")
for r in wf_details:
    vals = {k: r.get(k, 0) for k in methods}
    best_k = max(vals, key=vals.get)
    print(f"  [{r['window']:2d}] {r['test_start']}~{r['test_end']} "
          f"{r.get('tft_only',0):5.1f} {r.get('lgbm_only',0):5.1f} "
          f"{r.get('v1_lr_all',0):5.1f} {r.get('v2_lr_slide',0):5.1f} "
          f"{r.get('v2_lgbm_slide',0):5.1f} {r.get('v2_lgbm_all',0):5.1f}  "
          f"{labels[best_k][:8]}")
print("="*85)

In [ ]:
## 앙상블 메타모델 v2 (개선 비교)

### v1 → v2 변경점
| | v1 | v2 |
|--|-----|-----|
| 피처 | tft_prob, lgbm_prob (2개) | + disagreement, consensus_strength, vol_5d, risk_flag (6개) |
| 학습 범위 | 전체 이력 (Window 0~i-1) | 최근 4윈도우 (슬라이딩) |
| 메타모델 | LogisticRegression | LR + LightGBM 비교 |

### 비교 대상 (6가지)
1. TFT 단독
2. LightGBM 단독
3. **v1**: LR + 2피처 + 전체이력
4. **v2-A+B**: LR + 6피처 + 슬라이딩
5. **v2-A+B+C**: LightGBM + 6피처 + 슬라이딩
6. **v2-A+C**: LightGBM + 6피처 + 전체이력

### 기존 v1 성능 (참고)
- v1 Stacking DA: 55.21% (in-sample CV)
- v1 OOS DA: ~52-53% (백테스트에서 확인)
- 앙상블 백테스트 v6: +89.02%, Sharpe 0.985

## 앙상블 Walk-Forward (체크포인트 로드)\n\n### 구조\n```\n각 윈도우:\n  TFT 체크포인트 로드 (수초) → 예측\n  LightGBM 체크포인트 로드 (수초) → 예측\n  GARCH 체크포인트 로드 (수초) → 변동성\n  → 3가지 앙상블 방법 비교\n```\n\n### 앙상블 방법\n1. **Simple Average**: (TFT + LightGBM) / 2\n2. **Weighted Average**: TFT*0.6 + LightGBM*0.4\n3. **Stacking**: LogisticRegression (3-fold CV)\n\n### 비교 기준\n- TFT WF 평균 DA: 51.07%\n- LightGBM WF 평균 DA: 52.30%\n- 앙상블이 개별 모델보다 높으면 성공